In [21]:
import pandas as pd
import numpy as np
import unicodedata
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. Load Data (Bypass pybaseball)
# ---------------------------------------------------------
try:
    url_bat = "https://www.baseball-reference.com/data/war_daily_bat.txt"
    url_pitch = "https://www.baseball-reference.com/data/war_daily_pitch.txt"
    bwar_b = pd.read_csv(url_bat)
    bwar_p = pd.read_csv(url_pitch)
except Exception as e:
    print(f"❌ Data Load Error: {e}")

# ---------------------------------------------------------
# 2. Target Selection
# ---------------------------------------------------------
target_hitters_dict = {
    'Giancarlo Stanton': 'Exit Velocity', 'Tommy Pham': 'Chase% (Discipline)', 
    'Miguel Rojas': 'Batting Average', 'Paul Goldschmidt': 'Batting Average',
    'Andrew McCutchen': 'Chase% (Discipline)', 'Carlos Santana': 'Chase% (Discipline)',
    'Donovan Solano': 'Batting Average', 'Mark Canha': 'Chase% (Discipline)',
    'DJ LeMahieu': 'Chase% (Discipline)', 'Albert Pujols': 'Exit Velocity', 
    'Michael Brantley': 'Batting Average', 'Evan Longoria': 'Batting Average',
    'Josh Harrison': 'Batting Average', 'Brandon Belt': 'Batting Average',
    'Matt Carpenter': 'Batting Average'
} 

target_pitchers_dict = {
    'Aroldis Chapman': 'Fastball Velocity', 'Luis García': 'Chase% (Induction)',
    'Justin Verlander': 'Fastball Spin Rate', 'Justin Wilson': 'Fastball Velocity',
    'Huascar Brazobán': 'Fastball Velocity', 'Carlos Carrasco': 'Chase% (Induction)',
    'Brooks Raley': 'Fastball Spin Rate', 'Ryan Brasier': 'Chase% (Induction)',
    'Max Scherzer': 'Chase% (Induction)', 'Ryan Pressly': 'Chase% (Induction)',
    'Chris Sale': 'Chase% (Induction)', 'Blake Treinen': 'Chase% (Induction)',
    'Chris Martin': 'Chase% (Induction)', 'Sonny Gray': 'Fastball Spin Rate',
    'Charlie Morton': 'Fastball Velocity', 'Kirby Yates': 'Chase% (Induction)',
    'Joe Kelly': 'Fastball Velocity', 'Daniel Bard': 'Fastball Spin Rate',
    'Matt Bush': 'Fastball Velocity', 'Corey Kluber': 'Fastball Velocity'
}

# ---------------------------------------------------------
# 3. Improved Logic Function
# ---------------------------------------------------------
def clean_name(name_str):
    name_str = str(name_str).strip().lower()
    return unicodedata.normalize('NFKD', name_str).encode('ASCII', 'ignore').decode('utf-8')

def validate_war_with_survival(bwar_df, target_dict):
    results = []
    # 📍 유형별 생존 통계를 저장할 딕셔너리
    stats = {cat: {'total': 0, 'survived': 0} for cat in set(target_dict.values())}
    
    bwar_df['name_clean'] = bwar_df['name_common'].apply(clean_name)

    for name, category in target_dict.items():
        stats[category]['total'] += 1 # 분모: 일단 리스트에 있으면 무조건 카운트
        
        clean_target_name = clean_name(name)
        player_data = bwar_df[(bwar_df['name_clean'] == clean_target_name) & 
                              (bwar_df['year_ID'].isin([2022, 2023, 2024, 2025]))].copy()
        
        if player_data.empty:
            continue
            
        # 중복 제거 (이닝/타석 많은 선수 기준)
        if len(player_data['mlb_ID'].unique()) > 1:
            main_id = player_data.groupby('mlb_ID')['G'].sum().idxmax()
            player_data = player_data[player_data['mlb_ID'] == main_id]

        if 'team_ID' in player_data.columns:
            player_data = player_data[player_data['team_ID'] != 'TOT']

        # 📍 생존 판정 (2025년 출장 경기 수 기준)
        is_2025_active = (2025 in player_data['year_ID'].values)
        g_2025 = player_data[player_data['year_ID'] == 2025]['G'].sum() if is_2025_active else 0
        
        if is_2025_active and g_2025 > 5:
            stats[category]['survived'] += 1 # 분자: 5경기 이상 뛴 경우만 생존 인정
            
            p_age = player_data[player_data['year_ID'] == 2025]['age'].max()
            war_22 = player_data[player_data['year_ID'] == 2022]['WAR'].sum()
            war_23 = player_data[player_data['year_ID'] == 2023]['WAR'].sum()
            war_24 = player_data[player_data['year_ID'] == 2024]['WAR'].sum()
            actual_war_25 = player_data[player_data['year_ID'] == 2025]['WAR'].sum()
            
            # Trend Projection
            if (war_22 > war_23) and (war_23 > war_24):
                proj_25 = (war_24 * 0.8) + (war_23 * 0.2)
            elif (war_22 < war_23) and (war_23 < war_24):
                proj_25 = (war_24 * 0.8) + (war_23 * 0.2)
            else:
                proj_25 = ((war_24 * 5) + (war_23 * 4) + (war_22 * 3)) / 12
                
            results.append({
                'Player': name,
                'Age': int(p_age) if not np.isnan(p_age) else 0,
                'Type': category,
                'Proj_WAR': round(proj_25, 2),
                'Actual_WAR': round(actual_war_25, 2),
                'Surplus': round(actual_war_25 - proj_25, 2)
            })
        else:
            # 5경기 미만이면 결과 테이블에는 안 나오지만, 위에서 total은 이미 올라감 (생존율 하락)
            print(f"🛑 Skipped: '{name}' (Retired or Inactive in 2025)")

    # 결과 정리
    df_res = pd.DataFrame(results).sort_values(by='Surplus', ascending=False) if results else pd.DataFrame()
    
    summary = []
    for cat, val in stats.items():
        rate = (val['survived'] / val['total'] * 100) if val['total'] > 0 else 0
        summary.append({'Type': cat, 'Total': val['total'], 'Survived': val['survived'], 'Rate(%)': round(rate, 1)})
    
    return df_res, pd.DataFrame(summary)

# ---------------------------------------------------------
# 4. Final Output
# ---------------------------------------------------------
h_res, h_sum = validate_war_with_survival(bwar_b, target_hitters_dict)
p_res, p_sum = validate_war_with_survival(bwar_p, target_pitchers_dict)

print("\n📊 [TYPE-BASED SURVIVAL RATE]")
print("-" * 60)
print(pd.concat([h_sum, p_sum]).sort_values(by='Rate(%)', ascending=False).to_string(index=False))

print("\n🔥 [DETAILED RESULTS]")
print("-" * 60)
print(h_res.to_string(index=False))
print(p_res.to_string(index=False))

🛑 Skipped: 'Albert Pujols' (Retired or Inactive in 2025)
🛑 Skipped: 'Michael Brantley' (Retired or Inactive in 2025)
🛑 Skipped: 'Evan Longoria' (Retired or Inactive in 2025)
🛑 Skipped: 'Josh Harrison' (Retired or Inactive in 2025)
🛑 Skipped: 'Brandon Belt' (Retired or Inactive in 2025)
🛑 Skipped: 'Matt Carpenter' (Retired or Inactive in 2025)
🛑 Skipped: 'Joe Kelly' (Retired or Inactive in 2025)
🛑 Skipped: 'Daniel Bard' (Retired or Inactive in 2025)
🛑 Skipped: 'Matt Bush' (Retired or Inactive in 2025)
🛑 Skipped: 'Corey Kluber' (Retired or Inactive in 2025)

📊 [TYPE-BASED SURVIVAL RATE]
------------------------------------------------------------
               Type  Total  Survived  Rate(%)
Chase% (Discipline)      5         5    100.0
 Chase% (Induction)      9         9    100.0
 Fastball Spin Rate      4         3     75.0
  Fastball Velocity      7         4     57.1
      Exit Velocity      2         1     50.0
    Batting Average      8         3     37.5

🔥 [DETAILED RESULTS]
---